# rlmflow coding-agent walkthrough

Build a recursive coding agent, run a task, and produce the trace the rest of these notebooks consume. This is the **source of `examples/data/notebook-coding-agent/`** — workspace, session, trace, and the generated boids artifact all live there.

The other two notebooks read from the same directory:

- [`node_basics.ipynb`](./node_basics.ipynb) — querying the trace: `walk`, `find`, `where`, `diff`, `session.chain_to`, …
- [`viz_walkthrough.ipynb`](./viz_walkthrough.ipynb) — visualizing the trace: `tree`, `gantt`, `code_log`, mermaid / dot / d2, `report_md`, the inline Gradio viewer, etc.

Running this notebook end-to-end **regenerates** the canonical example. Skip ahead to either of the others if you just want to consume the saved trace — they work offline against what's already committed.

This file mirrors `examples/coding-agent/agent.py` (same workspace, runtime, tools, `RLMFlow` config) but as a notebook so you can poke at every step.

## 1. Build the agent

Four pieces snap together:

- `Workspace` — durable, branchable storage for sessions, traces, and any files the agent writes.
- `Runtime` — where REPL code runs. `LocalRuntime` runs in-process; `DockerRuntime` runs each step in a container.
- Tools — plain Python functions registered on the runtime. `FILE_TOOLS` gives the agent `read_file`, `write_file`, `edit_file`, `ls`, `grep`, etc.
- `RLMFlow` — wires an LLM client (and optional cheaper alternates) to the runtime + workspace, and exposes `start` / `step` / `fork`.

In [7]:
from pathlib import Path

from rlmflow.llm import OpenAIClient
from rlmflow.rlm import RLMConfig, RLMFlow
from rlmflow.runtime.local import LocalRuntime
from rlmflow.tools import FILE_TOOLS
from rlmflow.workspace import Workspace

# Canonical example dir — node_basics.ipynb and viz_walkthrough.ipynb both
# read from here. Running this notebook live overwrites it.
WORKSPACE_DIR = Path("../data/notebook-coding-agent").resolve()


def build_agent(
    workspace_dir: str | Path = WORKSPACE_DIR,
    max_depth: int = 2,
    max_iterations: int = 30,
) -> RLMFlow:
    """Construct a coding agent identical to examples/coding-agent/agent.py."""
    workspace = Workspace.create(Path(workspace_dir).resolve())
    runtime = LocalRuntime(workspace=workspace)
    runtime.register_tools(FILE_TOOLS)

    return RLMFlow(
        llm_client=OpenAIClient("gpt-5"),
        runtime=runtime,
        workspace=workspace,
        config=RLMConfig(max_depth=max_depth, max_iterations=max_iterations),
        llm_clients={
            "fast": {
                "model": OpenAIClient("gpt-5-mini"),
                "description": "Cheap model for smaller subtasks",
            },
        },
    )

## 2. Run a task

`agent.start(query)` returns the initial `Node`. Every call to `agent.step(node)` then advances exactly one tick — one LLM call, one REPL execution, or a wait-resolution — and returns a new immutable `Node`. That sequence of nodes *is* the run.

`live_view()` is a context-manager that opens a self-updating Rich tree; you own the loop and just hand it the latest state on each tick. The agent loop and the visualization are decoupled — swap `live_view()` for any other display, or drop it entirely with no change to the loop.

In [8]:
from rlmflow.utils.viz import live_view

TASK = """Create a simple boids simulation in plain html and javascript. Render 100s of fast moving and coloful boids. Do not add configurations -- just the canvas. The speed should be a constant, not based on size. Split each component into separate files and delegate the work. Make sure to run and test the components after they are done. The main runnable interface should be in index.html. Save in output/boids-simulation."""

agent = build_agent(max_depth=2)
state = agent.start(TASK)
states = [state]

In [9]:
print(agent.build_system_prompt(state))

## Role

You are a recursive agent with a Python REPL. You solve tasks by writing and executing Python programs, and you can delegate subtasks to sub-agents with fresh context windows.

## REPL

- Every response is exactly one ```repl``` code block. Tools are already in the namespace.
- Variables persist across turns within one agent.
- `AGENT_ID`, `DEPTH`, `MAX_DEPTH` are set; cannot `delegate` when `DEPTH == MAX_DEPTH`.
- **Final answer:** call `done(answer)` exactly once when complete — that string is what the parent/user sees. No `done`, no result.
- **Execute, don't narrate.** Every turn runs code that makes progress.
- Output is truncated (~12k chars). Slice, summarize, or delegate — don't `print` huge values.

## Strategy

**Size up → search → delegate → combine.**

1. **Size up.** Measure long input first (`CONTEXT.info()`, `len(read_file(...))`).
2. **Search before solving.** Sample head/tail/middle, grep landmarks, inspect schema before committing.
3. **Delegate parallel work

In [10]:
from rlmflow.utils.viz import live_view

with live_view() as view:
    view(state)
    while not state.finished:
        state = agent.step(state)
        states.append(state)
        view(state)

print(f"{len(states)} states  ·  final: {state.agent_id} [{state.type}]")
print(f"query : {states[0].query[:120]!r}...")
print(f"result: {(state.result or '')[:200]}")

Output()

4 states  ·  final: root [result]
query : 'Create a simple boids simulation in plain html and javascript. Render 100s of fast moving and coloful boids. Do not add '...
result: Boids simulation written to output/boids-simulation with modular JS (boid, simulation, renderer) and index.html entrypoint. Verified imports and animation loop wiring.


In [13]:
from rlmflow.utils.trace import save_trace

trace_path = save_trace(states, agent.workspace.trace_dir, metadata={"task": TASK})
print(f"trace -> {trace_path}  ({len(states)} states)")

trace -> /Users/shyam/Code/rlmkit/examples/notebooks/notebook-coding-agent/trace/trace.json  (4 states)


In [14]:
import html as _html
import re
from IPython.display import HTML

# Build a self-contained page from the agent's index.html, then embed via
# <iframe srcdoc=...> (Jupyter doesn't resolve relative iframe URLs).
# Handles BOTH ES modules (recursive flatten + strip import/export) and plain
# <script src=...> tags, so it doesn't matter which style the agent picked.
candidates = sorted(agent.workspace.root.glob("output/**/index.html"))
if not candidates:
    raise FileNotFoundError(f"no index.html under {agent.workspace.root}")
html_path = candidates[-1]
base = html_path.parent


def _inline_css(m):
    p = base / m.group(1)
    return f"<style>\n{p.read_text()}\n</style>" if p.exists() else m.group(0)


def _inline_plain_js(m):
    p = base / m.group(1)
    return f"<script>\n{p.read_text()}\n</script>" if p.exists() else m.group(0)


def _flatten_module(entry: str) -> str:
    """DFS-flatten an ES-module graph into one inline <script>, deps first."""
    visited: set = set()
    chunks: list[str] = []

    def visit(path):
        path = path.resolve()
        if path in visited or not path.exists():
            return
        visited.add(path)
        src = path.read_text()
        for m in re.finditer(
            r'^\s*import\b.*?\bfrom\s+[\'"]([^\'"]+)[\'"]', src, flags=re.MULTILINE
        ):
            visit(path.parent / m.group(1))
        src = re.sub(
            r'^\s*import\b.*?\bfrom\s+[\'"][^\'"]+[\'"];?\s*$',
            "", src, flags=re.MULTILINE,
        )
        src = re.sub(r"^\s*export\s+default\s+", "", src, flags=re.MULTILINE)
        src = re.sub(r"^\s*export\s+", "", src, flags=re.MULTILINE)
        chunks.append(f"// === {path.name} ===\n{src}")

    visit(base / entry)
    return "<script>\n" + "\n".join(chunks) + "\n</script>"


content = html_path.read_text()
content = re.sub(r'<link\b[^>]*\bhref="([^"]+)"[^>]*/?>', _inline_css, content, flags=re.I)
# ES-module <script type="module" src=...> first (recursive), then any plain <script src=>.
content = re.sub(
    r'<script\b(?=[^>]*\btype="module")[^>]*\bsrc="([^"]+)"[^>]*>\s*</script>',
    lambda m: _flatten_module(m.group(1)), content, flags=re.I,
)
content = re.sub(
    r'<script\b[^>]*\bsrc="([^"]+)"[^>]*>\s*</script>',
    _inline_plain_js, content, flags=re.I,
)

print(f"rendering {html_path}")
HTML(
    f'<iframe srcdoc="{_html.escape(content, quote=True)}" '
    f'width="100%" height="600" '
    f'style="border:1px solid #ddd; border-radius:6px; background:#0a0a0f"></iframe>'
)

rendering /Users/shyam/Code/rlmkit/examples/notebooks/notebook-coding-agent/output/boids-simulation/index.html


## 3. Open the interactive viewer

`open_viewer` is the headline visualization — clickable graph of every node, step slider, and the full per-agent conversation. Pass `session=` so it can reconstruct the actual chat from disk; `states` alone only gives you what each snapshot captured.

For every other render (mermaid, gantt, code log, report, …) and the full tour, see [`viz_walkthrough.ipynb`](./viz_walkthrough.ipynb).

In [15]:
from rlmflow.utils.viewer import open_viewer

open_viewer(
    states,
    session=agent.workspace.session.root,
    inline=True,
    height=720,
    quiet=True,
)

## Next

The trace, session, and generated artifact are now committed under `examples/data/notebook-coding-agent/`. Pick up from there:

- [`node_basics.ipynb`](./node_basics.ipynb) — query the graph: `walk`, `find`, `where`, `path_to`, snapshot diffs, session reconstruction, event streaming.
- [`viz_walkthrough.ipynb`](./viz_walkthrough.ipynb) — render the run: trees, mermaid / dot / d2, gantt, code log, error summary, message stream, token / budget / report bundles.

Other ways in:

- Run the same agent from a terminal: `python examples/coding-agent/agent.py --workspace ./myproject`
- Render any saved trace from the CLI: `rlmflow render <trace-dir> -f mermaid-flowchart` / `report-md` / `gantt-html`
- Visualization roadmap: [`docs/internal/visualization_roadmap.md`](../../docs/internal/visualization_roadmap.md)